In [1]:
!wget -q https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip

In [2]:
!unzip -q scifact.zip

In [3]:
!find scifact -maxdepth 2 -type f

scifact/qrels/test.tsv
scifact/qrels/train.tsv
scifact/corpus.jsonl
scifact/queries.jsonl


In [4]:
import json
with open("scifact/corpus.jsonl", "r") as f:
    first_document = json.loads(f.readline())
print(first_document)

{'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the

In [5]:
corpus = {}
with open("scifact/corpus.jsonl", "r") as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc["_id"]] = {
            "title": doc["title"],
            "text": doc["text"]
        }
print("Number of documents:", len(corpus))
first_id = next(iter(corpus))
print("\nFirst document ID:", first_id)
print("Title:", corpus[first_id]["title"])

Number of documents: 5183

First document ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.


In [6]:
queries = {}
with open("scifact/queries.jsonl", "r") as f:
    for line in f:
        query = json.loads(line)
        queries[query["_id"]] = query["text"]
print("Number of queries:", len(queries))
first_query_id = next(iter(queries))
print("\nFirst query ID:", first_query_id)
print("Query:", queries[first_query_id])

Number of queries: 1109

First query ID: 0
Query: 0-dimensional biomaterials lack inductive properties.


In [7]:
import pandas as pd
qrels_train = pd.read_csv(
    "scifact/qrels/train.tsv",
    sep="\t"
)
qrels_test = pd.read_csv(
    "scifact/qrels/test.tsv",
    sep="\t"
)
print("Train qrels:", len(qrels_train))
print("Test qrels:", len(qrels_test))
print("\nTrain columns:")
print(qrels_train.columns.tolist())
print("\nFirst train qrel:")
print(qrels_train.iloc[0])

Train qrels: 919
Test qrels: 339

Train columns:
['query-id', 'corpus-id', 'score']

First train qrel:
query-id            0
corpus-id    31715818
score               1
Name: 0, dtype: int64


In [8]:
corpus_ids = set(corpus.keys())
train_corpus_ids = set(qrels_train["corpus-id"].astype(str))
test_corpus_ids = set(qrels_test["corpus-id"].astype(str))
missing_train_docs = train_corpus_ids - corpus_ids
missing_test_docs = test_corpus_ids - corpus_ids
print("Missing train documents:", len(missing_train_docs))
print("Missing test documents:", len(missing_test_docs))

Missing train documents: 0
Missing test documents: 0


In [9]:
query_ids = set(queries.keys())
train_query_ids = set(qrels_train["query-id"].astype(str))
test_query_ids = set(qrels_test["query-id"].astype(str))
print("Missing train queries:",
      len(train_query_ids - query_ids))
print("Missing test queries:",
      len(test_query_ids - query_ids))

Missing train queries: 0
Missing test queries: 0


In [10]:
documents = []
for doc_id, doc in corpus.items():
  documents.append({
      "doc_id":str(doc_id),
      "title":doc["title"],
      "text":doc["text"]
  })
documents_df = pd.DataFrame(documents)
print("Number of documents:",len(documents_df))
print("\nColumns:")
print(documents_df.columns.tolist())
display(documents_df.head())

Number of documents: 5183

Columns:
['doc_id', 'title', 'text']


,doc_id,title,text
0,4983,Microstructural development of human newborn c...,Alterations of the architecture of cerebral wh...
1,5836,Induction of myelodysplasia by myeloid-derived...,Myelodysplastic syndromes (MDS) are age-depend...
2,7912,"BC1 RNA, the transcript from a master gene for...",ID elements are short interspersed elements (S...
3,18670,The DNA Methylome of Human Peripheral Blood Mo...,DNA methylation plays an important role in bio...
4,19238,The human myelin basic protein gene is include...,Two human Golli (for gene expressed in the oli...


In [11]:
print("Missing values:")
print(documents_df.isnull().sum())

Missing values:
doc_id    0
title     0
text      0
dtype: int64


In [12]:
print("\nEmpty titles:",(documents_df["title"].str.strip()=="").sum())
print("Empty texts:",(documents_df["text"].str.strip()=="").sum())


Empty titles: 0
Empty texts: 0


In [13]:
documents_df["word_count"] = (
    documents_df["text"].str.split().str.len()
)
print(documents_df["word_count"].describe())

count    5183.000000
mean      201.810920
std        86.007594
min        26.000000
25%       147.000000
50%       192.000000
75%       247.000000
max      1524.000000
Name: word_count, dtype: float64


In [14]:
documents_df["content"]=(documents_df["title"]+"\n\n"+documents_df["text"])
print(documents_df[["doc_id","content"]].head(1).to_string(index=False))

doc_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [15]:
queries_df = pd.DataFrame([
    {
        "query_id": str(query_id),
        "query":query_text
    }
    for query_id,query_text in queries.items()
])
print("Number of queries:",len(queries_df))
display(queries_df.head())

Number of queries: 1109


,query_id,query
0,0,0-dimensional biomaterials lack inductive prop...
1,2,1 in 5 million in UK have abnormal PrP positiv...
2,4,1-1% of colorectal cancer patients are diagnos...
3,6,10% of sudden infant death syndrome (SIDS) dea...
4,9,32% of liver transplantation programs required...


In [16]:
test_qrels_df = qrels_test.copy()
test_qrels_df["query-id"] = (
    test_qrels_df["query-id"].astype(str)
)
test_qrels_df["corpus-id"] = (
    test_qrels_df["corpus-id"].astype(str)
)
print(test_qrels_df.head())

  query-id corpus-id  score
0        1  31715818      1
1        3  14717500      1
2        5  13734012      1
3       13   1606628      1
4       36   5152028      1


In [17]:
documents_df["word_count"].describe()

,word_count
count,5183.000000
mean,201.810920
std,86.007594
min,26.000000
25%,147.000000
50%,192.000000
75%,247.000000
max,1524.000000


In [18]:
documents_df.isnull().sum()

,0
doc_id,0
title,0
text,0
word_count,0
content,0


In [19]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded!")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
Embedding dimension: 384


/tmp/ipykernel_1328/2339020451.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [20]:
sample_text = documents_df["content"].iloc[0]
sample_embedding = embedding_model.encode(
    sample_text
)
print("Embedding shape:", sample_embedding.shape)
print("\nFirst 10 values:")
print(sample_embedding[:10])

Embedding shape: (384,)

First 10 values:
[ 0.06377757 -0.09766445 -0.01591938  0.06727889  0.01674544  0.02413773
 -0.05277781  0.01633654  0.0210111   0.02301066]


In [21]:
document_texts = documents_df["content"].tolist()
document_embeddings = embedding_model.encode(
    document_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)
print("Embedding matrix shape:", document_embeddings.shape)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Embedding matrix shape: (5183, 384)


In [22]:
print("Shape:", document_embeddings.shape)
print("Data type:", document_embeddings.dtype)
print("First vector:")
print(document_embeddings[0][:10])

Shape: (5183, 384)
Data type: float32
First vector:
[ 0.06377757 -0.09766445 -0.01591938  0.0672789   0.01674544  0.02413773
 -0.05277781  0.01633654  0.0210111   0.02301067]


In [23]:
import numpy as np
np.save(
    "document_embeddings.npy",
    document_embeddings
)
print("Embeddings saved!")

Embeddings saved!


In [24]:
embedding_norms = np.linalg.norm(
    document_embeddings,
    axis=1
)
print("Minimum norm:", embedding_norms.min())
print("Maximum norm:", embedding_norms.max())
print("Average norm:", embedding_norms.mean())

Minimum norm: 0.9999998
Maximum norm: 1.0000001
Average norm: 1.0


In [25]:
print(document_embeddings.shape)

(5183, 384)


In [26]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.4 MB/s eta 0:00:00


In [27]:
import faiss
document_embeddings = np.asarray(
    document_embeddings,
    dtype="float32"
)
embedding_dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dimension)
index.add(document_embeddings)

print("FAISS index created!")
print("Embedding dimension:", embedding_dimension)
print("Number of indexed documents:", index.ntotal)

FAISS index created!
Embedding dimension: 384
Number of indexed documents: 5183


In [28]:
doc_id_mapping = documents_df["doc_id"].tolist()
print("FAISS position 0 →", doc_id_mapping[0])
print("FAISS position 1 →", doc_id_mapping[1])
print("FAISS position 2 →", doc_id_mapping[2])

FAISS position 0 → 4983
FAISS position 1 → 5836
FAISS position 2 → 7912


In [29]:
def search_documents(query, top_k=5):
    """
    Search the FAISS index and return the top-k documents.
    """
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(
        query_embedding,
        top_k
    )
    results = []
    for score, idx in zip(scores[0], indices[0]):
        doc_id = doc_id_mapping[idx]
        document = documents_df[
            documents_df["doc_id"] == doc_id
        ].iloc[0]
        results.append({
            "doc_id": doc_id,
            "score": float(score),
            "title": document["title"],
            "text": document["text"]
        })

    return results

In [30]:
sample_query = queries_df["query"].iloc[0]
print("QUERY:")
print(sample_query)
results = search_documents(
    sample_query,
    top_k=5
)
print("\nTOP RESULTS:\n")
for i, result in enumerate(results, 1):
    print(f"Rank {i}")
    print("Document ID:", result["doc_id"])
    print("Similarity:", round(result["score"], 4))
    print("Title:", result["title"])

QUERY:
0-dimensional biomaterials lack inductive properties.

TOP RESULTS:

Rank 1
Document ID: 29638116
Similarity: 0.3453
Title: Complex Tissue and Disease Modeling using hiPSCs.
Rank 2
Document ID: 4346436
Similarity: 0.3333
Title: Nonlinear Elasticity in Biological Gels
Rank 3
Document ID: 17388232
Similarity: 0.3129
Title: Mechanical regulation of cell function with geometrically modulated elastomeric substrates
Rank 4
Document ID: 31715818
Similarity: 0.2879
Title: New opportunities: the use of nanotechnologies to manipulate and track stem cells.
Rank 5
Document ID: 927561
Similarity: 0.2871
Title: Emergent structures and dynamics of cell colonies by contact inhibition of locomotion


In [31]:
for i, result in enumerate(results, 1):
    print(
        f"{i}. {result['score']:.4f} | "
        f"{result['doc_id']} | "
        f"{result['title']}"
    )

1. 0.3453 | 29638116 | Complex Tissue and Disease Modeling using hiPSCs.
2. 0.3333 | 4346436 | Nonlinear Elasticity in Biological Gels
3. 0.3129 | 17388232 | Mechanical regulation of cell function with geometrically modulated elastomeric substrates
4. 0.2879 | 31715818 | New opportunities: the use of nanotechnologies to manipulate and track stem cells.
5. 0.2871 | 927561 | Emergent structures and dynamics of cell colonies by contact inhibition of locomotion


In [32]:
from collections import defaultdict
qrels_lookup = defaultdict(set)
for _, row in test_qrels_df.iterrows():
    if row["score"] > 0:
        qrels_lookup[row["query-id"]].add(
            row["corpus-id"]
        )
print("Number of test queries with relevance judgments:",
      len(qrels_lookup))

Number of test queries with relevance judgments: 300


In [33]:
def retrieve_doc_ids(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(
        query_embedding,
        top_k
    )
    return [
        doc_id_mapping[idx]
        for idx in indices[0]
    ]

In [34]:
query_id = test_qrels_df["query-id"].iloc[0]
query_text = queries_df[
    queries_df["query_id"] == query_id
]["query"].iloc[0]

retrieved_ids = retrieve_doc_ids(
    query_text,
    top_k=5
)
relevant_ids = qrels_lookup[query_id]
print("Query ID:", query_id)
print("\nRelevant documents:")
print(relevant_ids)
print("\nRetrieved documents:")
print(retrieved_ids)

Query ID: 1

Relevant documents:
{'31715818'}

Retrieved documents:
['29638116', '4346436', '3874000', '10786948', '31715818']


In [35]:
def recall_at_k(retrieved_ids, relevant_ids):
    retrieved_relevant = set(retrieved_ids) & set(relevant_ids)
    return len(retrieved_relevant) / len(relevant_ids)

In [36]:
recall = recall_at_k(
    retrieved_ids,
    relevant_ids
)
print("Recall@5:", round(recall, 4))

Recall@5: 1.0


In [37]:
query_text_lookup = dict(
    zip(
        queries_df["query_id"],
        queries_df["query"]
    )
)
print("Queries available:", len(query_text_lookup))

Queries available: 1109


In [38]:
test_query_ids = list(qrels_lookup.keys())
test_query_texts = [
    query_text_lookup[qid]
    for qid in test_query_ids
]
print("Test queries:", len(test_query_ids))

Test queries: 300


In [39]:
test_query_embeddings = embedding_model.encode(
    test_query_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")
print("Query embedding shape:")
print(test_query_embeddings.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Query embedding shape:
(300, 384)


In [40]:
scores, indices = index.search(
    test_query_embeddings,
    10
)
print("Scores shape:", scores.shape)
print("Indices shape:", indices.shape)

Scores shape: (300, 10)
Indices shape: (300, 10)


In [41]:
all_retrieved_ids = []
for row in indices:
    retrieved_ids = [
        doc_id_mapping[idx]
        for idx in row
    ]
    all_retrieved_ids.append(retrieved_ids)
print("Example retrieved IDs:")
print(all_retrieved_ids[0])

Example retrieved IDs:
['29638116', '4346436', '3874000', '10786948', '31715818', '17388232', '86129154', '927561', '19855358', '1769799']


In [42]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    return len(retrieved & relevant) / len(relevant)

In [43]:
recall_scores = {
    "Recall@1": [],
    "Recall@3": [],
    "Recall@5": [],
    "Recall@10": []
}
for qid, retrieved_ids in zip(
    test_query_ids,
    all_retrieved_ids
):
    relevant_ids = qrels_lookup[qid]
    for k in [1, 3, 5, 10]:
        metric_name = f"Recall@{k}"
        score = recall_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )
        recall_scores[metric_name].append(score)

In [44]:
baseline_results = {
    metric: sum(values) / len(values)
    for metric, values in recall_scores.items()
}
for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

Recall@1: 0.4823
Recall@3: 0.6603
Recall@5: 0.7379
Recall@10: 0.7833


In [45]:
def reciprocal_rank(retrieved_ids, relevant_ids):
    relevant_ids = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0.0

In [46]:
rr_scores = []
for qid, retrieved_ids in zip(
    test_query_ids,
    all_retrieved_ids
):
    relevant_ids = qrels_lookup[qid]
    rr = reciprocal_rank(
        retrieved_ids,
        relevant_ids
    )
    rr_scores.append(rr)
mrr = sum(rr_scores) / len(rr_scores)
print("MRR:", round(mrr, 4))

MRR: 0.6047


In [47]:
baseline_results["MRR"] = mrr
baseline_df = pd.DataFrame(
    {
        "Metric": list(baseline_results.keys()),
        "Score": list(baseline_results.values())
    }
)
baseline_df["Score"] = baseline_df["Score"].round(4)
display(baseline_df)

,Metric,Score
0,Recall@1,0.4823
1,Recall@3,0.6603
2,Recall@5,0.7379
3,Recall@10,0.7833
4,MRR,0.6047


In [48]:
baseline_df.to_csv(
    "baseline_retrieval_results.csv",
    index=False
)
print("Baseline results saved.")

Baseline results saved.


In [49]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.9 MB/s eta 0:00:00


In [50]:
from groq import Groq
print("Groq library imported successfully!")

Groq library imported successfully!


In [51]:
from google.colab import userdata
groq_api_key = userdata.get("GROQ_API_KEY")
if groq_api_key:
    print("✅ Groq API key loaded successfully.")
else:
    print("❌ Groq API key not found.")

✅ Groq API key loaded successfully.


In [52]:
client = Groq(
    api_key=groq_api_key
)
print("Groq client initialized.")

Groq client initialized.


In [56]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Key loaded:", key is not None)
print("Key length:", len(key) if key else 0)

Key loaded: True
Key length: 56


In [58]:
from groq import Groq
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Say hello"}
    ],temperature=0
)
print(response.choices[0].message.content)

Hello! 👋 How can I assist you today?


In [59]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful scientific assistant."
        },
        {
            "role": "user",
            "content": "What is retrieval-augmented generation?"
        }
    ],
    temperature=0
)
print(response.choices[0].message.content)

**Retrieval‑Augmented Generation (RAG)** is a family of neural‑language‑model architectures that combine **information retrieval** with **text generation** so that a model can “look up” external documents at inference time and condition its output on the retrieved evidence.  

Below is a compact scientific overview that covers the motivation, core components, typical variants, advantages, challenges, and practical tips for building or using a RAG system.

---

## 1. Why augment generation with retrieval?

| Problem in vanilla generation | How retrieval helps |
|------------------------------|----------------------|
| **Stale or limited knowledge** – a pretrained LM only knows what was in its training data (often months/years old). | The model can fetch up‑to‑date facts from a dynamic corpus (e.g., Wikipedia, news, internal docs). |
| **Hallucinations** – LM may fabricate plausible‑sounding but false statements. | Evidence grounding forces the model to copy or paraphrase real text, redu

In [60]:
def build_context(results):
    context_parts = []
    for i, result in enumerate(results, 1):
        context_parts.append(
            f"[Document {i}]\n"
            f"Title: {result['title']}\n"
            f"Content: {result['text']}"
        )
    return "\n\n".join(context_parts)

In [61]:
context = build_context(results)
print(context[:3000])

[Document 1]
Title: Complex Tissue and Disease Modeling using hiPSCs.
Content: Defined genetic models based on human pluripotent stem cells have opened new avenues for understanding disease mechanisms and drug screening. Many of these models assume cell-autonomous mechanisms of disease but it is possible that disease phenotypes or drug responses will only be evident if all cellular and extracellular components of a tissue are present and functionally mature. To derive optimal benefit from such models, complex multicellular structures with vascular components that mimic tissue niches will thus likely be necessary. Here we consider emerging research creating human tissue mimics and provide some recommendations for moving the field forward.

[Document 2]
Title: Nonlinear Elasticity in Biological Gels
Content: Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological function of some tissues, ha

In [62]:
def rag_answer(query, top_k=5):
    retrieved_docs = search_documents(
        query,
        top_k=top_k
    )
    context = build_context(retrieved_docs)
    system_prompt = """
You are a scientific question-answering assistant.

Answer the user's question using only the provided context.

Rules:
- Do not use outside knowledge.
- If the context does not contain enough information, say so.
- Do not invent facts.
- Give a concise answer.
"""

    user_prompt = f"""
Context:

{context}

Question:
{query}

Answer:
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )
    return {
        "answer": response.choices[0].message.content,
        "retrieved_documents": retrieved_docs
    }

In [63]:
sample_query = queries_df["query"].iloc[0]
rag_result = rag_answer(
    sample_query,
    top_k=5
)
print("QUESTION")
print("-" * 80)
print(sample_query)
print("\nANSWER")
print(rag_result["answer"])

QUESTION
--------------------------------------------------------------------------------
0-dimensional biomaterials lack inductive properties.

ANSWER
The provided documents do not contain information about the inductive properties of 0‑dimensional biomaterials, so I cannot answer that statement based on the given context.


In [64]:
print("\nRETRIEVED DOCUMENTS")
print("-" * 80)
for i, doc in enumerate(
    rag_result["retrieved_documents"],
    1
):
    print(f"\nRank {i}")
    print("Score:", round(doc["score"], 4))
    print("Title:", doc["title"])


RETRIEVED DOCUMENTS
--------------------------------------------------------------------------------

Rank 1
Score: 0.3453
Title: Complex Tissue and Disease Modeling using hiPSCs.

Rank 2
Score: 0.3333
Title: Nonlinear Elasticity in Biological Gels

Rank 3
Score: 0.3129
Title: Mechanical regulation of cell function with geometrically modulated elastomeric substrates

Rank 4
Score: 0.2879
Title: New opportunities: the use of nanotechnologies to manipulate and track stem cells.

Rank 5
Score: 0.2871
Title: Emergent structures and dynamics of cell colonies by contact inhibition of locomotion


In [65]:
print(response.usage)

CompletionUsage(completion_tokens=3072, prompt_tokens=89, total_tokens=3161, completion_time=6.617882063, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=170), prompt_time=0.005046634, prompt_tokens_details=None, queue_time=0.023761314, total_time=6.622928697)


In [66]:
import time

In [67]:
def rag_answer_with_metrics(
    query,
    top_k=5,
    model="openai/gpt-oss-120b"
):
    total_start = time.perf_counter()
    retrieved_docs = search_documents(
        query,
        top_k=top_k
    )

    retrieval_end = time.perf_counter()
    context = build_context(retrieved_docs)
    system_prompt = """
You are a scientific question-answering assistant.

Answer the user's question using only the provided context.

Rules:
- Do not use outside knowledge.
- If the context does not contain enough information, say so.
- Do not invent facts.
- Give a concise answer.
"""

    user_prompt = f"""
Context:

{context}

Question:
{query}

Answer:
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    generation_end = time.perf_counter()
    usage = response.usage

    input_tokens = getattr(
        usage,
        "prompt_tokens",
        None
    )

    output_tokens = getattr(
        usage,
        "completion_tokens",
        None
    )

    total_tokens = getattr(
        usage,
        "total_tokens",
        None
    )

    retrieval_latency = (
        retrieval_end - total_start
    )

    generation_latency = (
        generation_end - retrieval_end
    )

    total_latency = (
        generation_end - total_start
    )
    return {
        "query": query,
        "answer": response.choices[0].message.content,
        "retrieved_documents": retrieved_docs,

        "model": model,

        "retrieval_latency_sec": retrieval_latency,
        "generation_latency_sec": generation_latency,
        "total_latency_sec": total_latency,

        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens
    }

In [68]:
sample_query = queries_df["query"].iloc[0]
result = rag_answer_with_metrics(
    sample_query,
    top_k=5
)
print("QUESTION:")
print(result["query"])
print("\nANSWER:")
print(result["answer"])
print("\n--- METRICS ---")
print("Model:", result["model"])
print("Retrieval latency:",
      round(result["retrieval_latency_sec"], 4), "seconds")
print("Generation latency:",
      round(result["generation_latency_sec"], 4), "seconds")
print("Total latency:",
      round(result["total_latency_sec"], 4), "seconds")
print("Input tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])
print("Total tokens:", result["total_tokens"])

QUESTION:
0-dimensional biomaterials lack inductive properties.

ANSWER:
The provided documents do not contain information about whether 0‑dimensional biomaterials lack inductive properties.

--- METRICS ---
Model: openai/gpt-oss-120b
Retrieval latency: 0.0274 seconds
Generation latency: 0.6173 seconds
Total latency: 0.6447 seconds
Input tokens: 944
Output tokens: 187
Total tokens: 1131


In [69]:
metrics = {
    "Model": result["model"],
    "Retrieval Latency (s)": round(
        result["retrieval_latency_sec"], 4
    ),
    "Generation Latency (s)": round(
        result["generation_latency_sec"], 4
    ),
    "Total Latency (s)": round(
        result["total_latency_sec"], 4
    ),
    "Input Tokens": result["input_tokens"],
    "Output Tokens": result["output_tokens"],
    "Total Tokens": result["total_tokens"]
}
metrics

{'Model': 'openai/gpt-oss-120b',
 'Retrieval Latency (s)': 0.0274,
 'Generation Latency (s)': 0.6173,
 'Total Latency (s)': 0.6447,
 'Input Tokens': 944,
 'Output Tokens': 187,
 'Total Tokens': 1131}

In [70]:
def calculate_cost(
    input_tokens,
    output_tokens,
    input_price_per_1m=None,
    output_price_per_1m=None
):
    if input_price_per_1m is None or output_price_per_1m is None:
        return None
    input_cost = (
        input_tokens / 1_000_000
    ) * input_price_per_1m
    output_cost = (
        output_tokens / 1_000_000
    ) * output_price_per_1m
    return input_cost + output_cost

In [71]:
baseline_cost = calculate_cost(
    result["input_tokens"],
    result["output_tokens"]
)
print("Baseline cost:", baseline_cost)

Baseline cost: None


In [72]:
import pandas as pd
baseline_log = pd.DataFrame([{
    "query": result["query"],
    "model": result["model"],
    "retrieval_latency_sec": result["retrieval_latency_sec"],
    "generation_latency_sec": result["generation_latency_sec"],
    "total_latency_sec": result["total_latency_sec"],
    "input_tokens": result["input_tokens"],
    "output_tokens": result["output_tokens"],
    "total_tokens": result["total_tokens"]
}])
baseline_log

,query,model,retrieval_latency_sec,generation_latency_sec,total_latency_sec,input_tokens,output_tokens,total_tokens
0,0-dimensional biomaterials lack inductive prop...,openai/gpt-oss-120b,0.027415,0.617284,0.6447,944,187,1131


In [73]:
sample_query_ids = test_query_ids[:10]
baseline_results = []
for query_id in sample_query_ids:
    query = query_text_lookup[query_id]
    result = rag_answer_with_metrics(
        query,
        top_k=5
    )
    baseline_results.append(result)
    print(
        f"Completed {len(baseline_results)}/10 | "
        f"Latency: {result['total_latency_sec']:.2f}s | "
        f"Tokens: {result['total_tokens']}"
    )

Completed 1/10 | Latency: 0.90s | Tokens: 1428
Completed 2/10 | Latency: 0.75s | Tokens: 1224
Completed 3/10 | Latency: 26.93s | Tokens: 2503
Completed 4/10 | Latency: 19.83s | Tokens: 2632
Completed 5/10 | Latency: 0.73s | Tokens: 2700
Completed 6/10 | Latency: 16.89s | Tokens: 2478
Completed 7/10 | Latency: 18.75s | Tokens: 2537
Completed 8/10 | Latency: 33.62s | Tokens: 1636
Completed 9/10 | Latency: 0.54s | Tokens: 1355
Completed 10/10 | Latency: 26.71s | Tokens: 2208


In [74]:
baseline_df = pd.DataFrame([
    {
        "query": r["query"],
        "model": r["model"],
        "retrieval_latency_sec": r["retrieval_latency_sec"],
        "generation_latency_sec": r["generation_latency_sec"],
        "total_latency_sec": r["total_latency_sec"],
        "input_tokens": r["input_tokens"],
        "output_tokens": r["output_tokens"],
        "total_tokens": r["total_tokens"]
    }
    for r in baseline_results
])
baseline_df

,query,model,retrieval_latency_sec,generation_latency_sec,total_latency_sec,input_tokens,output_tokens,total_tokens
0,0-dimensional biomaterials show inductive prop...,openai/gpt-oss-120b,0.026842,0.876369,0.903211,1147,281,1428
1,"1,000 genomes project enables mapping of genet...",openai/gpt-oss-120b,0.095676,0.653079,0.748754,1040,184,1224
2,1/2000 in UK have abnormal PrP positivity.,openai/gpt-oss-120b,0.055260,26.871607,26.926867,2304,199,2503
3,5% of perinatal mortality is due to low birth ...,openai/gpt-oss-120b,0.023959,19.809871,19.833830,2430,202,2632
4,A deficiency of vitamin B12 increases blood le...,openai/gpt-oss-120b,0.025351,0.705129,0.730480,2454,246,2700
5,A high microerythrocyte count raises vulnerabi...,openai/gpt-oss-120b,0.029920,16.862560,16.892481,2245,233,2478
6,"A total of 1,000 people in the UK are asymptom...",openai/gpt-oss-120b,0.040339,18.706786,18.747125,2361,176,2537
7,ADAR1 binds to Dicer to cleave pre-miRNA.,openai/gpt-oss-120b,0.025246,33.597216,33.622462,1466,170,1636
8,AIRE is expressed in some skin tumors.,openai/gpt-oss-120b,0.022560,0.515983,0.538543,1224,131,1355
9,ALDH1 expression is associated with better bre...,openai/gpt-oss-120b,0.023639,26.689257,26.712896,2056,152,2208


In [75]:
print("Average retrieval latency:",
      round(
          baseline_df["retrieval_latency_sec"].mean(),4
      ))
print("Average generation latency:",
      round(
          baseline_df["generation_latency_sec"].mean(),4
      ))
print("Average total latency:",
      round(
          baseline_df["total_latency_sec"].mean(),4
      ))
print("Average input tokens:",
      round(
          baseline_df["input_tokens"].mean(),2
      ))
print("Average output tokens:",
      round(
          baseline_df["output_tokens"].mean(),2
      ))
print("Average total tokens:",
      round(
          baseline_df["total_tokens"].mean(),2
      ))

Average retrieval latency: 0.0369
Average generation latency: 14.5288
Average total latency: 14.5657
Average input tokens: 1872.7
Average output tokens: 197.4
Average total tokens: 2070.1


In [76]:
p95_latency = baseline_df["total_latency_sec"].quantile(0.95)
print(
    "P95 latency:",
    round(p95_latency, 4),
    "seconds"
)

P95 latency: 30.6094 seconds
